# Practical 06 — RAG using a Pre-trained LLM
Rubric-aligned workflow: document cleaning, chunking, embeddings, Chroma vector database, retrieval, LLM generation and LLM-only comparison. Run cells to produce actual outputs.

## 1. Install packages

In [1]:
%pip -q install chromadb sentence-transformers transformers torch

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


## 2. Imports and sample domain documents

In [2]:
import re
import torch
import chromadb

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Choose GPU when available; otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)

documents = [
    ("RAG overview",
     "RAG combines information retrieval with text generation. It searches a document collection and provides useful passages to an LLM before it writes an answer."),
    ("RAG components",
     "A RAG system has a retriever and a generator. The retriever finds passages related to a question. The generator uses the question and retrieved passages to prepare an answer."),
    ("Chunking",
     "Document chunking splits long documents into smaller pieces. A small overlap can preserve context when information crosses chunk boundaries."),
    ("Embeddings",
     "Text embeddings are numerical vectors representing text meaning. Cosine similarity compares a question vector with document vectors and ranks passages by semantic similarity."),
    ("LLM limitations",
     "A pre-trained LLM may not know a newly supplied document and may provide details not supported by the source."),
    ("Retrieval quality",
     "RAG can ground answers in supplied documents. If retrieval selects unrelated or incomplete passages, the answer may still be weak or incorrect."),
    ("Evaluation",
     "Answer evaluation can consider relevance, evidence support, completeness, missing facts and unsupported claims."),
    ("Chroma vector database",
     "Chroma is a vector database used to store document chunks, their embeddings and metadata. When a question is asked, Chroma compares the question embedding with stored embeddings and returns the most similar document chunks. These retrieved chunks are then passed to the language model as context.")
]

def clean_text(text):
    # Remove extra spaces and unwanted characters without changing the meaning
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s.,;:!?()%-]", "", text)
    return text.strip()

documents = [(source, clean_text(text)) for source, text in documents]
print("Documents loaded:", len(documents))

Running on: cpu
Documents loaded: 8


## 3. Chunking and embeddings
Chunks use a word limit and overlap. These are starter settings and can be adjusted for longer documents.

In [3]:
def chunk_text(text, chunk_size=80, overlap=15):
    words=text.split(); result=[]; start=0
    while start<len(words):
        end=min(start+chunk_size,len(words)); result.append(" ".join(words[start:end]))
        if end==len(words): break
        start=end-overlap
    return result
chunks=[]
for source,text in documents:
    for j,part in enumerate(chunk_text(text)):
        chunks.append({"id":f"{source.replace(' ','_')}_{j}","source":source,"index":j,"text":part})
embedding_model=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
vectors=embedding_model.encode([c["text"] for c in chunks],normalize_embeddings=True)
print("Chunk count:",len(chunks),"Embedding shape:",vectors.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chunk count: 8 Embedding shape: (8, 384)


## 4. Chroma vector database
Chroma stores chunk text, vector embeddings and source metadata. The collection is configured for cosine distance.

In [4]:
client=chromadb.Client()
try: client.delete_collection("rag_practical")
except Exception: pass
collection=client.create_collection("rag_practical",metadata={"hnsw:space":"cosine"})
collection.add(ids=[c["id"] for c in chunks],
               documents=[c["text"] for c in chunks],
               metadatas=[{"source":c["source"],"chunk":c["index"]} for c in chunks],
               embeddings=vectors.tolist())
print("Indexed chunks:",collection.count())

Indexed chunks: 8


## 5. Retrieval and pre-trained LLM

In [5]:
def retrieve(question, top_k=3):
    q = embedding_model.encode(
        [question], normalize_embeddings=True
    ).tolist()

    result = collection.query(
        query_embeddings=q,
        n_results=min(top_k, collection.count())
    )

    return [
        {"text": d, "meta": m, "distance": float(dist)}
        for d, m, dist in zip(
            result["documents"][0],
            result["metadatas"][0],
            result["distances"][0]
        )
    ]


# FLAN-T5-Large generally follows instructions better than the Base version.
# If your Colab runtime runs out of memory, change this to "google/flan-t5-base".
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

llm = llm.to(device)
llm.eval()

print("Loaded model:", model_name)
print("Running on:", device)


def generate(prompt, max_new_tokens=120):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.inference_mode():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            do_sample=False,
            no_repeat_ngram_size=2,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded model: google/flan-t5-base
Running on: cpu


## 6. Complete RAG pipeline and baseline

In [6]:
def answer_rag(question, top_k=3):
    evidence = retrieve(question, top_k)

    # Keep source labels in the evidence display, but pass clean passage text
    # to the generator so it does not simply repeat a document title.
    context = "\n\n".join(
        f"Passage {i}: {item['text']}"
        for i, item in enumerate(evidence, start=1)
    )

    prompt = (
        "You are a careful question-answering assistant. "
        "Answer the question using the context passages below. "
        "Use the exact facts and terms stated in the passages. "
        "Write one or two complete sentences. "
        "Do not repeat passage labels or document titles. "
        "Do not add information that is not supported by the passages. "
        "If the passages do not contain the answer, say: "
        "'The provided documents do not contain enough information.'"
        f"\n\nContext passages:\n{context}"
        f"\n\nQuestion: {question}"
        "\nAnswer:"
    )

    return generate(prompt), evidence


def answer_llm_only(question):
    prompt = (
        "Answer the following question directly using your general knowledge. "
        "Give a clear, factual answer in one or two complete sentences. "
        "Do not use an unrelated analogy. If you are unsure, say so."
        f"\n\nQuestion: {question}"
        "\nAnswer:"
    )
    return generate(prompt)


question = "What are the two main parts of a RAG system?"

rag, evidence = answer_rag(question)
baseline = answer_llm_only(question)

print("Question:", question)
print("\nRAG:", rag)
print("\nLLM only:", baseline)
print("\nRetrieved evidence:")
for item in evidence:
    print(
        f"- Source: {item['meta']['source']} | "
        f"Distance: {item['distance']:.4f} | "
        f"Text: {item['text']}"
    )

Question: What are the two main parts of a RAG system?

RAG: RAG combines information retrieval with text generation.

LLM only: RAG is a system of sensors that detect and track the movement of objects in the environment.

Retrieved evidence:
- Source: RAG components | Distance: 0.4339 | Text: A RAG system has a retriever and a generator. The retriever finds passages related to a question. The generator uses the question and retrieved passages to prepare an answer.
- Source: RAG overview | Distance: 0.4890 | Text: RAG combines information retrieval with text generation. It searches a document collection and provides useful passages to an LLM before it writes an answer.
- Source: Retrieval quality | Distance: 0.5328 | Text: RAG can ground answers in supplied documents. If retrieval selects unrelated or incomplete passages, the answer may still be weak or incorrect.


## 7. Multi-question comparison and manual evaluation

In [7]:
questions = [
    "What are the two main parts of a RAG system?",
    "Why are documents split into chunks?",
    "What does an embedding represent?",
    "How does Chroma support retrieval?",
    "What can happen when retrieved passages are unrelated?"
]

results = []

for q in questions:
    rag_answer, evidence = answer_rag(q)
    baseline_answer = answer_llm_only(q)

    results.append({
        "question": q,
        "rag": rag_answer,
        "baseline": baseline_answer,
        "evidence": evidence
    })

    print("\n" + "-" * 90)
    print("Question:", q)
    print("\nRAG:", rag_answer)
    print("\nLLM-only:", baseline_answer)
    print("\nRetrieved evidence:")
    for item in evidence:
        print(
            f'- {item["meta"]["source"]} '
            f'(distance={item["distance"]:.4f}): {item["text"]}'
        )


------------------------------------------------------------------------------------------
Question: What are the two main parts of a RAG system?

RAG: RAG combines information retrieval with text generation.

LLM-only: RAG is a system of sensors that detect and track the movement of objects in the environment.

Retrieved evidence:
- RAG components (distance=0.4339): A RAG system has a retriever and a generator. The retriever finds passages related to a question. The generator uses the question and retrieved passages to prepare an answer.
- RAG overview (distance=0.4890): RAG combines information retrieval with text generation. It searches a document collection and provides useful passages to an LLM before it writes an answer.
- Retrieval quality (distance=0.5328): RAG can ground answers in supplied documents. If retrieval selects unrelated or incomplete passages, the answer may still be weak or incorrect.

------------------------------------------------------------------------------

## Result and conclusion
This notebook implements cleaning, chunking, embedding generation, Chroma indexing, top-k retrieval and generation with a pre-trained LLM. It compares the RAG answer with the same model without retrieved context. Record actual outputs and evaluation scores from your run; do not invent results.